# Notebook прямого эфира MicroDuck: навигация / веб-интерфейс RDK / развёртывание нескольких политик

Объединяет задачу, политику, ONNX, MuJoCo и приёмку RDK X5/BPU в единый воспроизводимый процесс.

## 0. Инструкция по запуску

Рекомендуется запускать Jupyter в Python-окружении `microduck-playground-stilts` (в исходном репозитории это каталог `02-可运行代码/microduck-playground-stilts`). Обученные модели и HBM для платы в Git не коммитятся; передаются через `MICRODUCK_ONNX`, `RDK_BPU_HBM` и `RDK_HOST`.

In [1]:
from pathlib import Path
import json
import os
import platform
import shlex
import socket
import subprocess


def locate_topic_root():
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / "02-可运行代码").is_dir() and (base / "01-任务资料").is_dir():  # каталоги upstream, не переводить
            return base
        group = base / "16-专题组队学习"  # каталог upstream, не переводить
        if group.is_dir():
            matches = list(group.glob("*/02-可运行代码"))  # см. комментарий выше
            if matches:
                return matches[0].parent
    configured = os.getenv("MICRODUCK_TOPIC_ROOT")
    if configured and Path(configured).is_dir():
        return Path(configured).resolve()
    raise RuntimeError("Не найден каталог темы; запустите Jupyter из 03-Notebook или задайте MICRODUCK_TOPIC_ROOT.")


TOPIC_ROOT = locate_topic_root()
PLAYGROUND_ROOT = TOPIC_ROOT / "02-可运行代码" / "microduck-playground-stilts"  # путь upstream, не переводить
OUTPUT_ROOT = TOPIC_ROOT / "03-Notebook" / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Каталог темы:", TOPIC_ROOT)
print("Исполняемый код:", PLAYGROUND_ROOT)
print("Выходной каталог:", OUTPUT_ROOT)
print("Станция Jupyter:", socket.gethostname(), platform.system(), platform.machine())
print("Адрес RDK:", os.getenv("RDK_HOST", "192.168.8.128"), "пользователь: sunrise")

专题目录: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题
可运行代码: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/02-可运行代码/microduck-playground-stilts
输出目录: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs
Jupyter 工作站: ubuntu Linux x86_64
RDK 地址: 192.168.8.128 用户: sunrise


## 1. Контракт модели и локальная проверка

Целевой контракт: actor `obs[1, 61] -> action[1, 14]`. Если для этой задачи в тему не загружены checkpoint или ONNX, Notebook явно покажет «модель не предоставлена» и не будет выдавать модель другой задачи за результат текущей.

In [2]:
TASK_ID = 'Mjlab-Velocity-Flat-MicroDuck'
MODEL_HINT = None
MODEL_PATH = Path(os.getenv("MICRODUCK_ONNX", "")) if os.getenv("MICRODUCK_ONNX") else None
if MODEL_PATH is None and MODEL_HINT:
    candidate = TOPIC_ROOT / MODEL_HINT
    if candidate.exists():
        MODEL_PATH = candidate
if MODEL_PATH is None and not MODEL_HINT and not TASK_ID:
    candidates = sorted(TOPIC_ROOT.glob("01-任务资料/**/*.onnx"))  # путь upstream, не переводить
    MODEL_PATH = candidates[0] if candidates else None
print("Задача:", TASK_ID or "интерфейс/веб-демо")
print("ONNX:", MODEL_PATH if MODEL_PATH else "не предоставлен")
if TASK_ID and MODEL_PATH is None:
    print("Для этой задачи в тему пока не загружен специальный ONNX; задайте MICRODUCK_ONNX и снова запустите проверку модели.")

任务: Mjlab-Velocity-Flat-MicroDuck
ONNX: 未提供
该任务尚未随专题提交任务专属 ONNX；请设置 MICRODUCK_ONNX 后再运行模型检查。


In [3]:
def inspect_onnx(path):
    try:
        import onnx
    except ImportError:
        print("Нет onnx: запустите Jupyter в окружении microduck-playground.")
        return None
    model = onnx.load(str(path))
    onnx.checker.check_model(model)
    def shape(value):
        return [d.dim_value if d.dim_value else (d.dim_param or "?") for d in value.type.tensor_type.shape.dim]
    result = {
        "inputs": [(item.name, shape(item)) for item in model.graph.input],
        "outputs": [(item.name, shape(item)) for item in model.graph.output],
        "nodes": len(model.graph.node),
        "metadata": {item.key: item.value for item in model.metadata_props},
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

contract = inspect_onnx(MODEL_PATH) if MODEL_PATH else None
print("PASS-model-contract-cell:", "проверка выполнена" if contract is not None else "модель не предоставлена, целевой контракт записан")

PASS-model-contract-cell: 模型未提供，已记录目标契约


## 2. Точка входа экспорта ONNX

Обучение даёт checkpoint политики; при экспорте нужно зафиксировать в ONNX и actor, и нормализацию наблюдений. Ниже — шаблон команды для этой задачи. `--video` записывает симуляционный повтор на локальной машине; полный видеофайл сохраняется в `outputs/` и не отправляется в Git.

```bash
uv run python scripts/export.py Mjlab-Velocity-Flat-MicroDuck\
  --checkpoint-file <CHECKPOINT.pt>\
  --onnx-file outputs/policy.onnx\
  --num-envs 1 --video --video-length 250
```

In [4]:
EXPORT_COMMAND = "python scripts/export_onnx.py run_dir=<FASTSAC_RUN_DIR> output=outputs/motrix_ball_balance_latest.onnx opset=11" if TASK_ID == "microduck-ball-balance" else ("uv run python scripts/export.py " + (TASK_ID or "<TASK_ID>") + " --checkpoint-file <CHECKPOINT.pt> --onnx-file outputs/policy.onnx --num-envs 1 --video --video-length 250")
print(EXPORT_COMMAND)
print("Проверка до и после экспорта: размерность наблюдений, порядок действий, нормализация и clip действий.")

uv run python scripts/export.py Mjlab-Velocity-Flat-MicroDuck --checkpoint-file <CHECKPOINT.pt> --onnx-file outputs/policy.onnx --num-envs 1 --video --video-length 250
导出前后检查：观测维度、动作顺序、归一化和 action clip。


## 3. Путь обучения: с нуля или с предобученного checkpoint

В каждом Notebook вход обучения и вход воспроизведения разделены: можно использовать `resume=False` как инициализацию с нуля либо `resume=True`, задав checkpoint конкретной задачи и состояния actor, critic и оптимизатора для дообучения. Демонстрационное видео по умолчанию использует уже проверенную модель задачи и не должно описываться как модель, обученная в текущей ячейке.

«10 итераций» здесь — это 10 циклов сэмплирования и оптимизации политики, а не 10 шагов обучения на каждое действие; например, `64 параллельных среды × 24 управляющих шага × 10` дают около `15360` переходов (transitions). Notebook навигации намеренно не обучает модель, а сразу подключает готовую; в Notebook возмущений и подъёма по лестнице тоже указано, какая задача-специфичная модель или официальная пара моделей используется.

## 3. Политика навигации: прямое подключение готовой модели

Ячейки навигации и развёртывания не переобучают модель, а сразу используют готовый ONNX. По умолчанию читается `outputs/walking_demo_latest.onnx` в каталоге этого Notebook; через `MICRODUCK_NAV_ONNX` можно указать другую уже обученную навигационную политику.

In [5]:
NAV_ONNX = os.getenv("MICRODUCK_NAV_ONNX", "")
default_nav_onnx = OUTPUT_ROOT / "walking_demo_latest.onnx"
legacy_nav_onnx = OUTPUT_ROOT / "microduck_demo_latest.onnx"
if NAV_ONNX:
    MODEL_PATH = Path(NAV_ONNX).expanduser().resolve()
elif default_nav_onnx.exists():
    MODEL_PATH = default_nav_onnx
elif legacy_nav_onnx.exists():
    MODEL_PATH = legacy_nav_onnx
print("Политика навигации подключена:", MODEL_PATH if MODEL_PATH and MODEL_PATH.exists() else "не найдена")
if MODEL_PATH is None or not MODEL_PATH.exists():
    print("Задайте MICRODUCK_NAV_ONNX=/path/to/navigation.onnx и запустите эту ячейку снова.")

导航策略直接接入: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/microduck_demo_latest.onnx


## 4. Локальный базовый инференс ONNX

Этот шаг только проверяет, что граф ONNX работает на этой машине, а выходы имеют правильную форму и конечные значения; это не оценка качества обучения и не приёмка BPU.

In [6]:
def onnx_local_smoke(path):
    try:
        import numpy as np
        import onnxruntime as ort
    except ImportError as exc:
        print("Нет локальных зависимостей для инференса:", exc)
        return None
    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    feeds = {}
    for item in session.get_inputs():
        shape = [dim if isinstance(dim, int) and dim > 0 else 1 for dim in item.shape]
        feeds[item.name] = np.zeros(shape, dtype=np.float32)
    outputs = session.run(None, feeds)
    info = {"providers": session.get_providers(), "inputs": [(x.name, x.shape) for x in session.get_inputs()], "outputs": [(x.name, x.shape) for x in session.get_outputs()], "finite": all(np.isfinite(x).all() for x in outputs)}
    print(json.dumps(info, ensure_ascii=False, indent=2, default=str))
    return outputs

if MODEL_PATH:
    local_outputs = onnx_local_smoke(MODEL_PATH)
else:
    print("Пропуск: ONNX для этой задачи ещё не предоставлен.")

{
  "providers": [
    "CPUExecutionProvider"
  ],
  "inputs": [
    [
      "obs",
      [
        1,
        61
      ]
    ]
  ],
  "outputs": [
    [
      "actions",
      [
        1,
        14
      ]
    ]
  ],
  "finite": true
}


## 5. Видео инференса на GPU

Эта ячейка сначала создаёт видео инференса на GPU, затем — видео RDK X5 BPU ниже. Ubuntu отвечает за физику и рендеринг MuJoCo/MotrixSim; актор политики выполняет экспортированный ONNX-граф через Torch на CUDA. Здесь показываются только «инференс на GPU» и результат задачи, конкретная модель видеокарты не раскрывается. Видеофайлы записываются в `outputs/gpu_videos/`.

Кроме воспроизведения возмущений, где явно используется политика ходьбы, каждая задача использует собственный ONNX: актор баскетбола с LSTM, актор FastSAC для баланса на шаре, для воспроизведения лестницы на GPU — два актора climber и get-up.

In [7]:
import sys

GPU_TASK = 'walking'
GPU_VIDEO_STEPS = int(os.getenv("MICRODUCK_GPU_VIDEO_STEPS", "250"))
GPU_VIDEO_OUTPUT = OUTPUT_ROOT / "gpu_videos" / f"{GPU_TASK}_gpu_latest.mp4" if GPU_TASK else None
GPU_ONNX_DEFAULTS = {
    "basketball": "/home/ubuntu/workspaces/microduck_bpu_models/converted/basketball.opset11.onnx",
    "walking": "/home/ubuntu/workspaces/microduck_bpu_models/converted/walking.opset11.onnx",
    "perturbation": "/home/ubuntu/workspaces/microduck_bpu_models/converted/walking.opset11.onnx",
    "stilt": "/home/ubuntu/workspaces/microduck_bpu_models/converted/stilts25.opset11.onnx",
    "swing": "/home/ubuntu/workspaces/microduck_bpu_models/converted/swing.opset11.onnx",
    "ball_balance": "/home/ubuntu/workspaces/microduck_bpu_models/converted/motrix_ball_balance.opset11.onnx",
}
GPU_ONNX = Path(os.getenv("MICRODUCK_GPU_ONNX", GPU_ONNX_DEFAULTS.get(GPU_TASK, ""))).expanduser() if GPU_TASK else None
GPU_LADDER_CLIMBER = Path(os.getenv("MICRODUCK_GPU_LADDER_CLIMBER", "/home/ubuntu/workspaces/microduck_bpu_models/converted/official_ladder_climber.opset11.onnx")).expanduser()
GPU_LADDER_GETUP = Path(os.getenv("MICRODUCK_GPU_LADDER_GETUP", "/home/ubuntu/workspaces/microduck_bpu_models/converted/official_ladder_getup.opset11.onnx")).expanduser()
MOTRIX_ROOT = Path(os.getenv("MOTRIX_ROOT", "/home/ubuntu/workspaces/MotrixLab")).expanduser() if GPU_TASK == "ball_balance" else None
MOTRIX_PYTHON = Path(os.getenv("MOTRIX_PYTHON", str(MOTRIX_ROOT / ".venv/bin/python"))).expanduser() if MOTRIX_ROOT else None
MOTRIX_RUNNER = PLAYGROUND_ROOT / "scripts" / "motrix_runner.py" if GPU_TASK == "ball_balance" else None
GPU_SCRIPT = PLAYGROUND_ROOT / "scripts" / ("motrix_gpu_inloop_video.py" if GPU_TASK == "ball_balance" else "gpu_inloop_video.py")
GPU_LADDER_SCRIPT = PLAYGROUND_ROOT / "scripts" / "bpu_official_ladder_pair_video.py"

if GPU_TASK == "ladder":
    gpu_ready = GPU_LADDER_SCRIPT.is_file() and GPU_LADDER_CLIMBER.is_file() and GPU_LADDER_GETUP.is_file()
    gpu_cmd = [
        sys.executable, str(GPU_LADDER_SCRIPT), "--backend", "gpu",
        "--official-root", "/home/ubuntu/workspaces/microduck-playground-official/experiments/desk-climb",
        "--climber-onnx", str(GPU_LADDER_CLIMBER), "--getup-onnx", str(GPU_LADDER_GETUP),
        "--output", str(GPU_VIDEO_OUTPUT), "--seconds", "24", "--num-envs", "64", "--record-env", "17",
    ]
else:
    gpu_ready = GPU_SCRIPT.is_file() and GPU_ONNX is not None and GPU_ONNX.is_file()
    gpu_cmd = [sys.executable, str(GPU_SCRIPT)] if GPU_TASK != "ball_balance" else [str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "video", str(GPU_SCRIPT)]
    if GPU_TASK != "ball_balance":
        gpu_cmd += ["--task", GPU_TASK]
    gpu_cmd += ["--onnx", str(GPU_ONNX), "--output", str(GPU_VIDEO_OUTPUT), "--steps", str(GPU_VIDEO_STEPS)]

print("Задача инференса на GPU:", GPU_TASK)
print("Модель GPU:", GPU_ONNX if GPU_TASK != "ladder" else (GPU_LADDER_CLIMBER, GPU_LADDER_GETUP))
if not gpu_ready:
    print("Воспроизведение на GPU пока не запускается: проверьте ONNX задачи, скрипт и окружение Motrix.")
elif GPU_TASK == "ball_balance" and (MOTRIX_PYTHON is None or not MOTRIX_PYTHON.is_file()):
    print("Не найдено окружение Python MotrixLab:", MOTRIX_PYTHON)
else:
    GPU_VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    print("Начинаем генерацию видео инференса на GPU.")
    gpu_result = subprocess.run(gpu_cmd, cwd=PLAYGROUND_ROOT, env=os.environ.copy(), capture_output=True, text=True)
    safe_lines = [line for line in (gpu_result.stdout + "\n" + gpu_result.stderr).splitlines() if "GPU" in line or "PASS-" in line or "Video" in line]
    print("\n".join(safe_lines[-20:]))
    if gpu_result.returncode != 0 or not GPU_VIDEO_OUTPUT.is_file():
        raise RuntimeError("Не удалось создать видео инференса на GPU; посмотрите лог скрипта.")
    GPU_REPORT = GPU_VIDEO_OUTPUT.with_suffix(".json")
    print("PASS-GPU-VIDEO:", GPU_VIDEO_OUTPUT)
    if GPU_REPORT.is_file():
        print(json.loads(GPU_REPORT.read_text(encoding="utf-8")))
    from IPython.display import Video, display
    display(Video(str(GPU_VIDEO_OUTPUT), embed=True))

GPU 推理任务: walking
GPU 模型: /home/ubuntu/workspaces/microduck_bpu_models/converted/walking.opset11.onnx
开始生成 GPU 推理视频。


GPU 策略推理已启用；不展示具体显卡型号。
PASS-GPU-VIDEO: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/gpu_videos/walking_gpu_latest.mp4
PASS-GPU-REPORT: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/gpu_videos/walking_gpu_latest.json
PASS-GPU-VIDEO: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/gpu_videos/walking_gpu_latest.mp4
{'task': 'walking', 'task_id': 'Mjlab-Velocity-Flat-MicroDuck', 'display_name': 'navigation', 'backend': 'GPU ONNX via Torch CUDA', 'policy_device': 'GPU', 'onnx_file': '/home/ubuntu/workspaces/microduck_bpu_models/converted/walking.opset11.onnx', 'onnx_sha256': '2da8200426614e4a83ef736f80e1ecfc1ca5c11e5dd28efd3524111d03314174', 'observation_dim': 61, 'action_dim': 14, 'recurrent': False, 'steps': 250, 'terminated_or_truncated_resets': 0, 'reset_steps': [], 'first_termination_step'

GIF 预览：../01-任务资料/精选视频/assets/notebook-replays/walking_gpu.gif；原始 MP4 未嵌入 Notebook，以控制仓库体积。

## 6. Видео BPU-in-the-loop на RDK X5

Эта ячейка показывает только новый замкнутый MP4 с платы и не читает эталонные GIF или ключевые кадры. Ubuntu выполняет физику MuJoCo и рендеринг EGL; на каждом управляющем шаге наблюдение задачи отправляется на RDK, RDK вычисляет 14-мерное действие на BPU с HBM текущей задачи и возвращает его на Ubuntu. Поэтому видео — настоящий замкнутый цикл: действия вычисляет BPU на плате, а картинку симуляции — Ubuntu.

Перед запуском автоматически перезапускается сервис политики задач на RDK. При переключении Notebook эту ячейку нужно выполнять заново, чтобы HBM на плате соответствовал текущей задаче.

In [8]:
import hashlib
import sys

BPU_TASK = 'walking'
BPU_LADDER_LOCAL_OPT_IN = os.getenv("MICRODUCK_RUN_LOCAL_LADDER_BPU", "0") == "1"
BPU_VIDEO_STEPS = int(os.getenv("MICRODUCK_BPU_VIDEO_STEPS", "250"))
BPU_VIDEO_WIDTH = int(os.getenv("MICRODUCK_BPU_VIDEO_WIDTH", "640"))
BPU_VIDEO_HEIGHT = int(os.getenv("MICRODUCK_BPU_VIDEO_HEIGHT", "480"))
BPU_CAMERA_DISTANCE = os.getenv("MICRODUCK_BPU_CAMERA_DISTANCE", "1.2")
BPU_CAMERA_ELEVATION = os.getenv("MICRODUCK_BPU_CAMERA_ELEVATION", "-8")
BPU_CAMERA_AZIMUTH = os.getenv("MICRODUCK_BPU_CAMERA_AZIMUTH", "90")
BPU_PORT = int(os.getenv("MICRODUCK_BPU_PORT", "8765"))
RDK_CONTROL_PORT = int(os.getenv("MICRODUCK_BPU_CONTROL_PORT", "8766"))
RDK_HOST = os.getenv("RDK_HOST", "192.168.8.128")
BPU_HBM_DEFAULTS = {
    "ball_balance": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/ball_balance/model_output/ball_balance.opset11.bin",
    "basketball": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/basketball/model_output/basketball.opset11.bin",
    "walking": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/walking/model_output/walking.opset11.bin",
    "perturbation": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/walking/model_output/walking.opset11.bin",
    "stilt": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/stilts25/model_output/stilts25.opset11.bin",
    "swing": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/swing/model_output/swing.opset11.bin",
    "ladder": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/ladder/model_output/ladder.opset11.bin",
}
RDK_HBM_DEFAULTS = {
    "ball_balance": "/home/sunrise/microduck_policy/hbm/ball_balance.bin",
    "basketball": "/home/sunrise/microduck_policy/hbm/basketball.bin",
    "walking": "/home/sunrise/microduck_policy/hbm/walking.bin",
    "perturbation": "/home/sunrise/microduck_policy/hbm/walking.bin",
    "stilt": "/home/sunrise/microduck_policy/hbm/stilts25.bin",
    "swing": "/home/sunrise/microduck_policy/hbm/swing.bin",
    "ladder": "/home/sunrise/microduck_policy/hbm/ladder.bin",
}
BPU_HBM = Path(os.getenv("MICRODUCK_BPU_HBM", BPU_HBM_DEFAULTS.get(BPU_TASK, ""))) if BPU_TASK else None
RDK_HBM = RDK_HBM_DEFAULTS.get(BPU_TASK, "") if BPU_TASK else ""
RDK_SERVER_SCRIPT = os.getenv("RDK_BPU_SERVER_SCRIPT", "/home/sunrise/microduck_policy/rdk_bpu_policy_server.py")
MOTRIX_ROOT = Path(os.getenv("MOTRIX_ROOT", "/home/ubuntu/workspaces/MotrixLab")).expanduser() if BPU_TASK == "ball_balance" else None
MOTRIX_PYTHON = Path(os.getenv("MOTRIX_PYTHON", str(MOTRIX_ROOT / ".venv/bin/python"))).expanduser() if MOTRIX_ROOT else None
MOTRIX_RUNNER = PLAYGROUND_ROOT / "scripts" / "motrix_runner.py" if BPU_TASK == "ball_balance" else None
BPU_VIDEO_SCRIPT = PLAYGROUND_ROOT / "scripts" / ("motrix_bpu_inloop_video.py" if BPU_TASK == "ball_balance" else "bpu_inloop_video.py")
BPU_VIDEO_OUTPUT = OUTPUT_ROOT / "bpu_videos" / f"{BPU_TASK}_bpu_latest.mp4" if BPU_TASK else None
ALLOW_CACHED_BPU = os.getenv("MICRODUCK_ALLOW_CACHED_BPU", "0") == "1"
BPU_LADDER_PAIR_SCRIPT = PLAYGROUND_ROOT / "scripts" / "bpu_official_ladder_pair_video.py"
BPU_LADDER_CLIMBER_HBM = Path(os.getenv("MICRODUCK_BPU_LADDER_CLIMBER_HBM", "/home/ubuntu/workspaces/microduck_bpu_models/compiled/official_ladder_climber/model_output/official_ladder_climber.bin")).expanduser()
BPU_LADDER_GETUP_HBM = Path(os.getenv("MICRODUCK_BPU_LADDER_GETUP_HBM", "/home/ubuntu/workspaces/microduck_bpu_models/compiled/official_ladder_getup/model_output/official_ladder_getup.bin")).expanduser()
BPU_LADDER_PAIR_OUTPUT = OUTPUT_ROOT / "bpu_videos" / "ladder_official_pair_bpu_latest.mp4"

def show_cached_bpu_video(path, reason):
    if not ALLOW_CACHED_BPU or path is None or not path.is_file():
        return False
    print("BPU-CACHED-VIDEO:", path)
    print("Пояснение:", reason)
    report = path.with_suffix(".json")
    if report.is_file():
        print(json.loads(report.read_text(encoding="utf-8")))
    from IPython.display import Video, display
    display(Video(str(path), embed=True))
    return True

def restart_rdk_policy_server():
    if not BPU_TASK or not RDK_HBM:
        return False
    try:
        if BPU_HBM is not None and BPU_HBM.is_file():
            payload = BPU_HBM.read_bytes()
            request = {
                "op": "upload_and_switch",
                "task": BPU_TASK,
                "size": len(payload),
                "sha256": hashlib.sha256(payload).hexdigest(),
            }
        else:
            payload = b""
            request = {"op": "switch", "task": BPU_TASK}
        with socket.create_connection((RDK_HOST, RDK_CONTROL_PORT), timeout=8) as control:
            control.sendall((json.dumps(request) + "\n").encode("utf-8"))
            if payload:
                for offset in range(0, len(payload), 1024 * 1024):
                    control.sendall(payload[offset:offset + 1024 * 1024])
            response = json.loads(control.makefile("rb").readline().decode("utf-8"))
        if response.get("ok"):
            action = "загружено и переключено" if payload else "переключено"
            print("Сервис политики задач RDK " + action + ":", BPU_TASK, "->", response.get("model"), "pid=", response.get("pid"))
            return True
        print("Сервис BPU-управления RDK отклонил переключение:", response)
    except (OSError, ValueError, json.JSONDecodeError) as exc:
        print("Сервис BPU-управления RDK недоступен, пробуем SSH:", exc)
    remote = "sunrise@" + RDK_HOST
    command = (
        "pkill -f '^python3 .*rdk_bpu_policy_server.py' >/dev/null 2>&1 || true; "
        "sleep 1; "
        "nohup python3 " + shlex.quote(RDK_SERVER_SCRIPT) +
        " --model " + shlex.quote(RDK_HBM) +
        " --port " + str(BPU_PORT) +
        " >/tmp/microduck_bpu_server.log 2>&1 </dev/null &"
    )
    result = subprocess.run(
        ["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=8", remote, "bash -lc " + shlex.quote(command)],
        capture_output=True, text=True,
    )
    if result.returncode:
        print("Не удалось запустить сервис политики RDK:", result.stderr.strip() or result.stdout.strip())
        return False
    print("Сервис политики задач RDK переключён:", BPU_TASK, "->", RDK_HBM)
    return True

if BPU_TASK == "ladder":
    pair_ready = BPU_LADDER_PAIR_SCRIPT.is_file() and BPU_LADDER_CLIMBER_HBM.is_file() and BPU_LADDER_GETUP_HBM.is_file()
    pair_cmd = [
        sys.executable, str(BPU_LADDER_PAIR_SCRIPT), "--backend", "bpu",
        "--climber-hbm", str(BPU_LADDER_CLIMBER_HBM), "--getup-hbm", str(BPU_LADDER_GETUP_HBM),
        "--output", str(BPU_LADDER_PAIR_OUTPUT), "--seconds", os.getenv("MICRODUCK_LADDER_SECONDS", "24"),
        "--num-envs", "64", "--record-env", "17",
    ]
    print("Официальная пара моделей BPU:", BPU_LADDER_CLIMBER_HBM, BPU_LADDER_GETUP_HBM)
    if not pair_ready:
        print("Официальное видео BPU для пары моделей лестницы пока не запускается: проверьте HBM climber/get-up и скрипт.")
    else:
        BPU_LADDER_PAIR_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        pair_result = subprocess.run(pair_cmd, cwd=PLAYGROUND_ROOT, env=os.environ.copy(), capture_output=True, text=True)
        print("\n".join((pair_result.stdout + "\n" + pair_result.stderr).splitlines()[-35:]))
        if pair_result.returncode != 0 or not BPU_LADDER_PAIR_OUTPUT.is_file():
            if not show_cached_bpu_video(BPU_LADDER_PAIR_OUTPUT, "Хост с платой сейчас недоступен; показано ранее полученное настоящее видео BPU с двумя моделями RDK X5."):
                raise RuntimeError("Не удалось создать видео BPU для официальной пары моделей лестницы.")
        else:
            print("PASS-BPU-VIDEO:", BPU_LADDER_PAIR_OUTPUT)
            pair_report = BPU_LADDER_PAIR_OUTPUT.with_suffix(".json")
            if pair_report.is_file():
                print(json.loads(pair_report.read_text(encoding="utf-8")))
            from IPython.display import Video, display
            display(Video(str(BPU_LADDER_PAIR_OUTPUT), embed=True))
elif not BPU_TASK:
    print("Для этой задачи нет проверяемого HBM, специфичного для неё; видео другой задачи вместо неё не используется.")
elif BPU_TASK == "ladder" and not BPU_LADDER_LOCAL_OPT_IN:
    print("Локальное видео HBM для 14-ступенчатой лестницы пропущено: это не официальная схема climber/get-up из сообщества.")
    print("Локальный эксперимент запускается только при явном MICRODUCK_RUN_LOCAL_LADDER_BPU=1; результат нельзя помечать как воспроизведение сообщества.")
elif BPU_HBM is None or not BPU_HBM.is_file():
    print("HBM на Ubuntu не существует:", BPU_HBM)
elif not BPU_VIDEO_SCRIPT.is_file():
    print("Не найден записывающий скрипт замкнутого цикла на плате:", BPU_VIDEO_SCRIPT)
elif BPU_TASK == "ball_balance" and (MOTRIX_PYTHON is None or not MOTRIX_PYTHON.is_file()):
    print("Не найдено окружение Python MotrixLab:", MOTRIX_PYTHON)
elif BPU_TASK == "ball_balance" and (MOTRIX_RUNNER is None or not MOTRIX_RUNNER.is_file()):
    print("Не найден запускатель MotrixLab:", MOTRIX_RUNNER)
elif restart_rdk_policy_server():
    BPU_VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    if BPU_TASK == "ball_balance":
        video_cmd = [str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "video", str(BPU_VIDEO_SCRIPT)]
    else:
        video_cmd = [sys.executable, str(BPU_VIDEO_SCRIPT)]
    if BPU_TASK != "ball_balance":
        video_cmd += ["--task", BPU_TASK]
    video_cmd += [
        "--bpu-host", RDK_HOST, "--bpu-port", str(BPU_PORT),
        "--hbm", str(BPU_HBM), "--output", str(BPU_VIDEO_OUTPUT),
        "--steps", str(BPU_VIDEO_STEPS), "--width", str(BPU_VIDEO_WIDTH),
        "--height", str(BPU_VIDEO_HEIGHT),
    ]
    video_cmd += [
        "--camera-distance", BPU_CAMERA_DISTANCE,
        "--camera-elevation", BPU_CAMERA_ELEVATION,
        "--camera-azimuth", BPU_CAMERA_AZIMUTH,
    ]
    print("Запуск генерации видео замкнутого цикла на плате:", " ".join(shlex.quote(x) for x in video_cmd))
    video_result = subprocess.run(
        video_cmd, cwd=PLAYGROUND_ROOT, env=os.environ.copy(),
        capture_output=True, text=True,
    )
    print("\n".join((video_result.stdout + "\n" + video_result.stderr).splitlines()[-35:]))
    if video_result.returncode != 0 or not BPU_VIDEO_OUTPUT.is_file():
        raise RuntimeError("Генерация видео BPU-in-the-loop не удалась; сначала посмотрите последние 35 строк лога выше.")
    BPU_REPORT = BPU_VIDEO_OUTPUT.with_suffix(".json")
    print("PASS-BPU-VIDEO:", BPU_VIDEO_OUTPUT)
    if BPU_REPORT.is_file():
        print(json.loads(BPU_REPORT.read_text(encoding="utf-8")))
    from IPython.display import Video, display
    display(Video(str(BPU_VIDEO_OUTPUT), embed=True))
else:
    show_cached_bpu_video(BPU_VIDEO_OUTPUT, "Хост с платой сейчас недоступен; показано ранее полученное настоящее видео задачи RDK X5 BPU.")

SIM_COMMAND = 'uv run python scripts/infer_policy.py --walking <WALKING.onnx> --new-cmd-obs'
print("Шаблон команды воспроизведения MuJoCo:", SIM_COMMAND)

RDK BPU 控制服务不可用，尝试 SSH: [Errno 113] No route to host


RDK 策略服务启动失败: ssh: connect to host 192.168.8.128 port 22: No route to host
BPU-CACHED-VIDEO: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/bpu_videos/walking_bpu_latest.mp4
说明: 本轮板端不可达，保留此前真实 RDK X5 BPU 任务视频。
{'task': 'walking', 'task_id': 'Mjlab-Velocity-Flat-MicroDuck', 'display_name': 'navigation', 'backend': 'RDK X5 BPU via hbm_runtime', 'bpu_host': '192.168.8.128', 'bpu_port': 8765, 'hbm_file': '/home/ubuntu/workspaces/microduck_bpu_models/compiled/walking/model_output/walking.opset11.bin', 'hbm_sha256': 'fc44b395300aa6b45e4b8e3677f669f6e275e5ce6d6c26e2a7c08c6e27129110', 'observation_dim_sent': 61, 'action_dim': 14, 'recurrent': False, 'source_root': '.', 'steps': 250, 'terminated_or_truncated_resets': 0, 'reset_steps': [], 'first_termination_step': None, 'first_termination_time_s': None, 'longest_uninterrupted_steps': 250, 'longest_uninterrupted_time_s': 5.0, 'video_nominal_time_s': 5.0, 'video_actual_time_s': 5.0

GIF 预览：../01-任务资料/精选视频/assets/notebook-replays/walking_bpu_cached.gif；原始 MP4 未嵌入 Notebook，以控制仓库体积。

MuJoCo 回放命令模板: uv run python scripts/infer_policy.py --walking <WALKING.onnx> --new-cmd-obs


## 7. Аудит RDK X5 и BPU

Предыдущая ячейка уже создала видео замкнутого цикла для задачи. Эта ячейка проверяет только связь с платой, метаданные HBM и логи сервиса; общий образец MobileNet доказывает лишь работоспособность рантайма системы и не может заменить видео задачи.

In [9]:
import time

RDK_HOST = os.getenv("RDK_HOST", "192.168.8.128")
BPU_SMOKE_MODEL = os.getenv(
    "RDK_BPU_SMOKE_MODEL",
    "/opt/tros/humble/lib/dnn_benchmark_example/config/X5/mobilenetv1_224x224_nv12.bin",
)
BPU_SMOKE_INPUT_BYTES = int(os.getenv("RDK_BPU_SMOKE_INPUT_BYTES", "75264"))
BPU_HBM = os.getenv("RDK_BPU_HBM", "")
print("Образец BPU по умолчанию:", BPU_SMOKE_MODEL)
print("Входные данные образца BPU по умолчанию:", BPU_SMOKE_INPUT_BYTES, "байт uint8")
print("HBM политики MicroDuck:", BPU_HBM or "для этой задачи не предоставлен")
probe_script = "for x in hrt_model_exec hb_mapper hrt_bin_dump hrt_bin_info python3; do if command -v $x >/dev/null 2>&1; then echo $x=$(command -v $x); else echo $x=MISSING; fi; done"
remote = "sunrise@" + RDK_HOST
probe = subprocess.run(["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=5", remote, "bash -lc " + shlex.quote(probe_script)], capture_output=True, text=True)
if probe.returncode:
    print("RDK пока недоступен, приёмка на плате пропущена:", probe.stderr.strip() or "ssh failed")
else:
    print("RDK подключён:", RDK_HOST)
    print(probe.stdout)
    if not BPU_HBM:
        print("Нет HBM задачи: общий образец вместо инференса задачи не выполняем.")
        smoke_model = BPU_SMOKE_MODEL
        smoke_bytes = BPU_SMOKE_INPUT_BYTES
        smoke_info_cmd = "hrt_model_exec model_info --model_file " + shlex.quote(smoke_model)
        smoke_info = subprocess.run(["ssh", remote, "bash -lc " + shlex.quote(smoke_info_cmd)], capture_output=True, text=True)
        print("Код выхода model_info образца BPU:", smoke_info.returncode)
        print(smoke_info.stdout[-4000:])
        print(smoke_info.stderr[-1000:])
        if smoke_info.returncode == 0:
            import numpy as np
            local_input = OUTPUT_ROOT / "rdk_bpu_runtime_smoke.bin"
            np.zeros(smoke_bytes, dtype=np.uint8).tofile(local_input)
            remote_dir = "/tmp/microduck_notebook"
            remote_input = remote_dir + "/rdk_bpu_runtime_smoke.bin"
            subprocess.run(["ssh", remote, "mkdir", "-p", remote_dir], check=True)
            subprocess.run(["scp", str(local_input), remote + ":" + remote_input], check=True)
            dump_dir = remote_dir + "/runtime_dump"
            infer_cmd = "rm -rf " + shlex.quote(dump_dir) + " && mkdir -p " + shlex.quote(dump_dir) + " && hrt_model_exec infer --model_file " + shlex.quote(smoke_model) + " --input_file " + shlex.quote(remote_input) + " --enable_dump true --dump_format txt --dump_path " + shlex.quote(dump_dir)
            started = time.perf_counter()
            infer = subprocess.run(["ssh", remote, "bash -lc " + shlex.quote(infer_cmd)], capture_output=True, text=True)
            elapsed_ms = (time.perf_counter() - started) * 1000.0
            print("BPU runtime sample exit:", infer.returncode, "round-trip ms:", round(elapsed_ms, 2))
            print(infer.stdout[-4000:])
            print(infer.stderr[-1000:])
            if infer.returncode == 0:
                print("PASS-rdk-bpu-runtime-sample: инференс образца X5 BPU вернул 0.")
    else:
        info_cmd = "hrt_model_exec model_info --model_file " + shlex.quote(BPU_HBM)
        info = subprocess.run(["ssh", remote, "bash -lc " + shlex.quote(info_cmd)], capture_output=True, text=True)
        print("HBM model_info exit:", info.returncode)
        print(info.stdout[-6000:])
        print(info.stderr[-2000:])
        if info.returncode == 0:
            # hrt_model_exec consumes binary tensors; this zero observation is
            # only a transport/BPU smoke input, not a task-quality evaluation.
            import numpy as np
            local_input = OUTPUT_ROOT / "zero_obs_61_f32.bin"
            np.zeros((1, 61), dtype=np.float32).tofile(local_input)
            remote_dir = "/tmp/microduck_notebook"
            remote_input = remote_dir + "/zero_obs_61_f32.bin"
            subprocess.run(["ssh", remote, "mkdir", "-p", remote_dir], check=True)
            subprocess.run(["scp", str(local_input), remote + ":" + remote_input], check=True)
            dump_dir = remote_dir + "/dump"
            infer_cmd = "rm -rf " + shlex.quote(dump_dir) + " && mkdir -p " + shlex.quote(dump_dir) + " && hrt_model_exec infer --model_file " + shlex.quote(BPU_HBM) + " --input_file " + shlex.quote(remote_input) + " --enable_dump true --dump_format txt --dump_path " + shlex.quote(dump_dir)
            started = time.perf_counter()
            infer = subprocess.run(["ssh", remote, "bash -lc " + shlex.quote(infer_cmd)], capture_output=True, text=True)
            elapsed_ms = (time.perf_counter() - started) * 1000.0
            print("BPU infer exit:", infer.returncode, "round-trip ms:", round(elapsed_ms, 2))
            print(infer.stdout[-6000:])
            print(infer.stderr[-2000:])
            if infer.returncode == 0:
                print("PASS-rdk-bpu: hrt_model_exec infer вернул 0.")
            else:
                print("Не пройдено: ориентируйтесь в первую очередь на отчёт BPU-in-the-loop из раздела 5.")

默认 BPU 样例模型: /opt/tros/humble/lib/dnn_benchmark_example/config/X5/mobilenetv1_224x224_nv12.bin
默认 BPU 样例输入: 75264 bytes uint8
MicroDuck 策略 HBM: 该任务尚未提供


RDK 暂不可达，跳过板端验收: ssh: connect to host 192.168.8.128 port 22: No route to host


## 8. Выводы для стрима

- `PASS-local-onnx`: проверка ONNX checker и локального ONNX Runtime пройдена.
- `PASS-GPU-VIDEO`: видео инференса на GPU создано и выполнено раньше видео BPU.
- `PASS-BPU-VIDEO`: каждое действие в видео Ubuntu MuJoCo вычислено на RDK X5 соответствующим HBM.
- `PASS-rdk-bpu-runtime-sample`: инференс установленного на плате образца X5 BPU завершился с кодом 0.
- `PASS-rdk-bpu`: SSH работает, на плате явно выбран BPU, команда инференса MicroDuck HBM завершилась с кодом 0, записаны формы входа и выхода и задержка.

Задача без `PASS-BPU-VIDEO` может считаться лишь «имеющимися материалами или описанием интерфейса», но не завершённым видео на плате.\n\nПравило статуса BPU: PASS-BPU-VIDEO означает текущий онлайн-запуск RDK. BPU-CACHED-VIDEO означает, что при офлайн-плате показано ранее полученное настоящее видео BPU; после появления связи с RDK запустите эту ячейку заново.